# imports

In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import pickle
import os
import cortex
#import neurosynth as ns
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
import importlib.util
import matplotlib.pyplot as plt

%matplotlib inline

In [2]:
# %config InlineBackend.figure_format = 'retina'

# set paths

In [3]:
annot_dir = '../../data/annotations_dfs/'
ep_model_dir = '../../data/models/episodes/'
ep_traj_dir = os.path.join(ep_model_dir, 't100_w50/', 'trajectories')
ep_seg_dir = os.path.join(ep_model_dir, 't100_w50/', 'events')
ep_emb_dir = os.path.join(ep_model_dir, 't100_w50/', 'embeddings', 'k:1000')
rec_model_dir = '../../data/models/participants/'
rec_traj_dir = os.path.join(rec_model_dir, 'trajectories')
rec_seg_dir = os.path.join(rec_model_dir, 'events', 'k:1000')
rec_emb_dir = os.path.join(rec_model_dir, 'embeddings', 'k:1000')
pickle_dir = '../../data/pickles/'
braindata_dir = '../../data/brains/'
fig_dir = '../../figures/'
brainfig_dir = os.path.join(fig_dir,'brains/')

# load data

In [23]:
with open(pickle_dir+'fit_models.p', 'rb') as f:
    models_dict = pickle.load(f)
    
brainweights = np.load(braindata_dir+'HuthModelWeights.npy')
mask = np.load(braindata_dir+'mask.npy')

away_from_topics = np.load(braindata_dir+'away_from_topic_vector.npy')
toward_topics = np.load(braindata_dir+'toward_topic_vector.npy')
    
#ns_dataset = ns.base.dataset.Dataset.load(pickle_dir+'neurosynth_dataset.p')

# episode events
episode_events = {episode : np.load(os.path.join(ep_seg_dir,episode+'_events.npy')) 
                  for episode in ['atlep1','atlep2','arrdev']}
# individual participant events
participant_events = {'atlep1' : [], 'atlep2' : [], 'arrdev' : [], 'delayed': [], 'prediction' : []}

for root, dirs, files in os.walk(rec_seg_dir):
    events_models = [f for f in files if f.startswith('debug') and f.endswith('events.npy')]
    for f_name in events_models:
        m = np.load(os.path.join(root,f_name))
        participant_events[os.path.split(root)[-1]].append((f_name.split('_')[0],m))
# episode-recall event mappings
event_mappings = dict()
for rectype in ['atlep1','atlep2','arrdev','delayed','prediction']:
    with open(os.path.join(rec_seg_dir,rectype,'event_mappings.p'), 'rb') as f:
        event_mappings[rectype] = pickle.load(f)
        
# session 1/session 2 psiturk ID mapppings
with open(os.path.join(pickle_dir,'id_maps.p'), 'rb') as f:
    id_maps = pickle.load(f)

## define some functions

In [28]:
def _z2r(z):
    """
    Function that calculates the inverse Fisher z-transformation

    Parameters
    ----------
    z : int or ndarray
        Fishers z transformed correlation value

    Returns
    ----------
    result : int or ndarray
        Correlation value

    """
    with np.errstate(invalid='ignore', divide='ignore'):
        return (np.exp(2 * z) - 1) / (np.exp(2 * z) + 1)

In [29]:
def _r2z(r):
    """
    Function that calculates the Fisher z-transformation

    Parameters
    ----------
    r : int or ndarray
        Correlation value

    Returns
    ----------
    result : int or ndarray
        Fishers z transformed correlation value

    """
    with np.errstate(invalid='ignore', divide='ignore'):
        return 0.5 * (np.log(1 + r) - np.log(1 - r))

In [30]:
def corr_mean(rs, axis=0):
    """
    Function that calculates the mean of correlation coefficients,
    performing Fisher z-transformation and inverse z-transormation
    
    Parameters
    ----------
    rs: : list or ndarray
        Correlation values
    
    Returns
    ----------
    result : float
        mean of correlation values

    """
    return _z2r(np.nanmean([_r2z(r) for r in rs], axis=axis))

# load neural speech representation semantic model 

In [6]:
# credit: https://github.com/HuthLab/speechmodeltutorial
spec = importlib.util.spec_from_file_location('HuthSpeechModel', '../HuthSpeechModel/SemanticModel.py')
HSM = importlib.util.module_from_spec(spec)
spec.loader.exec_module(HSM)
corpus = HSM.SemanticModel.load('../HuthSpeechModel/data/english1000sm.hf5')

In [7]:
# percent of words in episode corpus covered in Huth semantic model
for episode in models_dict.keys():
    if episode != 'delayed':
        nwords_covered = len([i for i in list(models_dict[episode]['cv'].vocabulary_) if i in corpus.vocab])
        print('{0}: {1}% coverage'.format(episode, nwords_covered/len(list(models_dict[episode]['cv'].vocabulary_))))

atlep1: 0.7756132756132756% coverage
atlep2: 0.7621178225205071% coverage
arrdev: 0.7450372208436724% coverage


# create weighted brain map for each topic dimension

In [7]:
# for episode, mods in models_dict.items():
#     if episode != 'delayed':
#         # get distribution of word weights for each topic
#         topic_wordweights = mods['tm'].components_ / mods['tm'].components_.sum(axis=1)[:, np.newaxis]

#         # voxel responses for each word in the corpus
#         word_activations = []
#         # for each word in the episode
#         for word in mods['cv'].vocabulary_:

#             # if it's in the Huth model's vocab, use its weights
#             if word in corpus.vocab:
#                 wordweight = corpus[word]
#             # otherwise fill with nans
#             else:
#                 wordweight = np.full(985, np.nan)

#             # get voxel-wise response for word weight
#             voxweights = np.dot(wordweight, brainweights)
#             word_activations.append(voxweights)
        
#         # get voxel weights for each topic dimensions
#         word_act_vals = np.nan_to_num(np.array(word_activations)
#         topic_voxweights =  np.dot(topic_wordweights, word_act_vals)
        
#         # save out weighted brainmaps
#         np.save(braindata_dir+'{}/topic_voxelweights.npy'.format(episode), topic_voxweights)
        
        

In [8]:
episode_topic_voxweights = {ep : np.load(os.path.join(braindata_dir, ep, 'topic_voxelweights.npy'))
                            for ep in ['atlep1','atlep2','arrdev']}


# predicted activation differences immediate to delayed

In [9]:
# most significant topics lost over delay
away_from_voxels = np.dot(away_from_topics, episode_topic_voxweights['atlep1'])

away_from_vol = cortex.Volume(away_from_voxels, 'S1', 'fullhead', mask=mask, cmap='RdBu_r')

In [10]:
# away_from_vol.save_nii(braindata_dir+'away_from_volume.nii.gz')

In [11]:
away_from_brainmap = cortex.webgl.show(away_from_vol, overlays_visible=(None, None))

Started server on port 23483
Stopping server


In [15]:
# away_from_brainmap.getImage(brainfig_dir+'away_from/away_from_inflated_trilinear_left_medial.png')

[{}]

In [12]:
# js_handle._set_view(**{'camera.azimuth': 90, 'camera.altitude': 40, 
#                        'camera.radius' : 300, 'surface.{subject}.sampler':'trilinear', 
#                     'surface.{subject}.unfold' : 0.5})
# animation = []
# for az, idx in zip([90, 180, 270, 360, 450], [0, .5, 1.0, 1.5, 2.0]):
#     animation.append({'state':'camera.azimuth', 'idx':idx, 'value':[az]})
    
# js_handle.makeMovie(animation)

In [13]:
# js_handle._capture_view()

In [17]:
# content most heavily re-focused after delay
toward_voxels = np.dot(toward_topics, episode_topic_voxweights['atlep1'])

toward_vol = cortex.Volume(toward_voxels, 'S1', 'fullhead', mask=mask, cmap='RdBu_r')

In [15]:
# toward_vol.save_nii(braindata_dir+'toward_volume.nii.gz')

In [18]:
toward_brainmap = cortex.webgl.show(toward_vol, overlays_visible=(None, None))

Started server on port 2471
Stopping server


In [22]:
# toward_brainmap.getImage(brainfig_dir+'away_from/toward_inflated_trilinear_left_medial.png')

[{}]

# what does a brain that is "remembering well" look like?

In [84]:
imm_goodrec_brains = np.empty((len(participant_events['atlep1']), 37226))
imm_badrec_brains = np.empty((len(participant_events['atlep1']), 37226))
del_goodrec_brains = np.empty((len(participant_events['atlep1']), 37226))
del_badrec_brains = np.empty((len(participant_events['atlep1']), 37226))

# for each participant and their immediat recall events
for i, (turkid1, imm_events) in enumerate(participant_events['atlep1']):
    
    # get their immediate recall event/episode event mappings
    imm_matches = next(ms for tid, ms in event_mappings['atlep1'] if tid == turkid1)
    
    # get their across-session ID
    sid, turkid2 = next((sid, id_maps[sid]['session 2']) for sid in id_maps.keys() 
                        if id_maps[sid]['session 1'] == turkid1)
    
    # get their delayed recall events and matches
    del_events = next(sub_evs for tid, sub_evs in participant_events['delayed'] if tid == turkid2)
    del_matches = next(ms for tid, ms in event_mappings['delayed'] if tid == turkid2)
    
    # find event-wise accuracy for immediate and delayed event recall
    imm_acc = np.diag((1-cdist(imm_events, episode_events['atlep1'][imm_matches], 'correlation')))
    del_acc = np.diag((1-cdist(del_events, episode_events['atlep1'][del_matches], 'correlation')))
    
    # compute weighted average topic vector and voxel activations for events remembered well...
    imm_goodrec = np.dot(imm_acc, episode_events['atlep1'][imm_matches]) / np.sum(imm_acc)
    imm_goodrec_brains[i] = np.dot(imm_goodrec, episode_topic_voxweights['atlep1'])
    
    del_goodrec = np.dot(del_acc, episode_events['atlep1'][del_matches]) / np.sum(del_acc)
    del_goodrec_brains[i] = np.dot(del_goodrec, episode_topic_voxweights['atlep1'])
    
    # ...and the opposite for events remembered poorly
    imm_badrec = np.dot(1-imm_acc, episode_events['atlep1'][imm_matches]) / np.sum(imm_acc)
    imm_badrec_brains[i] = np.dot(imm_badrec, episode_topic_voxweights['atlep1'])
    
    del_badrec = np.dot(1-del_acc, episode_events['atlep1'][del_matches]) / np.sum(del_acc)
    del_badrec_brains[i] = np.dot(del_badrec, episode_topic_voxweights['atlep1'])
    
goodbrain_i = imm_goodrec_brains.mean(axis=0)
badbrain_i = imm_badrec_brains.mean(axis=0)
goodbrain_d = del_goodrec_brains.mean(axis=0)
badbrain_d = del_badrec_brains.mean(axis=0)

In [85]:
goodbrain_i_vol = cortex.Volume(goodbrain_i, 'S1', 'fullhead', mask=mask, cmap='RdBu_r')
goodbrain_i_brainmap = cortex.webgl.show(goodbrain_i_vol, overlays_visible=(None, None))

Started server on port 41688


In [89]:
goodbrain_i_brainmap.getImage(brainfig_dir+'goodrec_i/goodbrain_i_inflated_trilinear_left_medial.png')

[{}]

Stopping server


In [97]:
braindiff_d = goodbrain_d - badbrain_d

In [98]:
braindiff_d_vol = cortex.Volume(braindiff_d, 'S1', 'fullhead', mask=mask, cmap='RdBu_r')
braindiff_d_brainmap = cortex.webgl.show(braindiff_d_vol, overlays_visible=(None, None))

Started server on port 31081


In [103]:
braindiff_d_brainmap.getImage(brainfig_dir+'braindiff_d/braindiff_d_inflated_trilinear_left_medial.png')

[{}]